# **Metabolic Health Explorer: preparación de datos para clasificar DIQ010**

## **1. Complemento a la Exploración inicial**

Dado que el análisis exploratorio inicial no fue lo suficientemente profundo para darnos un norte claro del tratamiento de los datos y la posible variable objetivo que buscamos predecir.

Es por esto que consideramos generar este Análisis exploratorio complementario al inicialmente realizado con la base original completa, de esta forma logramos dar más claridad a lo que buscamos predecir y cómo llegaremos a ello.

Cabe aclarar que profundizando en la información, habíamos accedido a un dataset incompleto, es por eso que buscamos la fuente original de diabetes y logramos identificarla como la base **'DIQ_H.xpt'** ubicada en la URL: https://wwwn.cdc.gov/nchs/nhanes/search/DataPage.aspx?Component=Questionnaire&CycleBeginYear=2013

### **1.1. Cargue de librerías para la exploración**

In [1]:
# librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para la visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

# Librerías para la manipulación de fechas
from datetime import datetime

# Otras librerías
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

### **1.2. Cargue de datos y exploración**

Cargamos la nueva base de datos encontrada con todas sus variables para realizar una exploración más exhaustiva y rigurosa de su contenido.

In [2]:
RUTA_RAW = Path("..") / "data" / "raw"

df_inicial = pd.read_sas(RUTA_RAW / "DIQ_H.xpt", format="xport")

df_inicial.shape

(9770, 54)

Podemos ver que la base de datos original que encontramos cuenta con:

- Observaciones: 9770
- Variables: 54

Es así que ahora contamos con una base de datos con mayor cantidad de información que podemos tratar y revisar con el fin de elegir una variable objetivo que tenga sentido con la pregunta de negocio planteada.

Siguiendo con la linea pleanteada inicialmente, revisaremos la cantidad de registros de Diabetes reportada:

In [3]:
df_inicial["DIQ010"].value_counts(dropna = False).sort_index()

DIQ010
1.0     737
2.0    8841
3.0     185
7.0       1
9.0       5
NaN       1
Name: count, dtype: int64

### **1.3. Posible Variable objetivo**

Consultando la información encontrada en la documentación de la base de datos (https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DIQ_H.htm), tenemos que los códigos que se encuentran en la base para la variable objetivo original responden a la siguiente pregunta: **¿Alguna vez le han dicho que tiene diabetes?**

Para este caso los participantes tenian las siguientes opciones:

| Código | Significado |
|---|---|
| 1 | Sí |
| 2 | No |
| 3 | Borderline |
| 7 | Rehusó a participar |
| 9 | No sabe si presenta diabetes |
| N/A | Sin dato |

Conforme a lo que buscamos, los registros enfocados en las respuestas 7. Rehusó a participar, 9. No sabe si presenta diabetes o N/A. Sin dato; no son clases relevantes para este estudio y no tendrían impacto en nuestro planteamiento inicial. 

Tras esta decision procedemos a eliminar esos registros:

In [4]:
filas_iniciales = len(df_inicial)

df_etapa1 = df_inicial[df_inicial["DIQ010"].isin([1, 2, 3])].copy()

Teniendo esta decisión en cuenta, tendremos los siguientes datos:

| Código | Significado | # Registros |
|---|---|---|
| 1 | Sí | 737 |
| 2 | No | 8841 |
| 3 | Borderline | 185 |

Es de resaltar que la base sigue presentando un desbalanceo bastante marcado, sin embargo al ser datos reales, es de esperar que sea así.

Adicionalmente, encontramos que existen **185 casos Borderline**, los cuales no son ni parte de la población que declara haber sido diagnosticados con diabétes pero si fueron informados que se encuentran en una etapa de **prediabetes**, es por esto que consideraremos los casos **Borderline** como casos positivos de diabetes.

Tras esta decisión, nuestro objetivo pasa a ser **"Predecir si un paciente está prediabetico o presenta diabetes"** sin recurrir a exámenes de laboratorio, de esta forma el negocio puede priorizar la toma de exámenes para confirmar el diagnóstico y posteriormente, generar planes preventivos para el paciente antes de llegar a una etapa de diabetes confirmada.

Creamos una variable binaria adicional de etiquetado llamada ``'Diabetes'``, con la siguiente clasificación:

| Código | Significado | Valor Etiqueta 'Diabetes' |
|---|---|---|
| 1 | Sí | 1 |
| 2 | No | 0 |
| 3 | Borderline | 1 |

In [5]:
df_etapa1['Diabetes'] = df_etapa1["DIQ010"].isin([1, 3]).astype(int)

print(df_etapa1.shape)

df_etapa1['Diabetes'].value_counts()
df_etapa1['Diabetes'].value_counts(normalize = True).round(3)

(9763, 55)


Diabetes
0    0.906
1    0.094
Name: proportion, dtype: float64

Es así que encontramos que tenemos la siguiente proporción de registros:

- **Reporta Diabetes negativa (0) = 90.6%**
- **Reporta Diabetes positiva o prediabetes (1) = 9.4%**

### **1.4. Nulos por variable**

Para una revisión más profunda, revisaremos la cantidad de nulos en el presente Dataset

In [6]:
nulos = df_etapa1.drop(columns="SEQN").set_index("Diabetes").isna()

conteo = nulos.groupby(level=0).sum().astype(int)
pct = nulos.groupby(level=0).mean().mul(100).round(1)

tabla_nulos = pd.concat(
    [conteo.T.add_prefix("NA Diabetes="), pct.T.add_prefix("% NA Diabetes=")],
    axis=1,
)
tabla_nulos

Diabetes,NA Diabetes=0,NA Diabetes=1,% NA Diabetes=0,% NA Diabetes=1
DIQ010,0,0,0.0,0.0
DID040,8841,185,100.0,20.1
DIQ160,2559,922,28.9,100.0
DIQ170,2559,740,28.9,80.3
DIQ172,2559,740,28.9,80.3
DIQ175A,7602,825,86.0,89.5
DIQ175B,8356,862,94.5,93.5
DIQ175C,8746,905,98.9,98.2
DIQ175D,8511,882,96.3,95.7
DIQ175E,8758,913,99.1,99.0


Encontramos que tenemos una gran cantidad de datos faltantes en cada una de las variables presentes, consultando la documentación encontramos que la razón por la que sucede es debido a que muchas de las respuestas de la encuesta están condicionadas a la respuesta de **Reporta Diabetes positiva (cat 1).**

Es así que encontramos que la base tiene un problema profundo de Data leakage:
- Las variables DID040, DIQ080, DIQ230 a DIQ360 solo cuentan con datos de las personas que respondieron Si en Diabetes, por lo que la mera presencia de datos en estas variables indica que la persona presenta diabetes.
- Las variables DIQ050 que representa 'Toma insulina en este momento' y DIQ070 que representa 'Toma pastillas para bajar la azucar en sangre' son variables consecuencia del diagnóstico de Diabetes, no son la causa de esta.
- Para las variables DIQ170, DIQ172 y DIQ180 podemos identificar el comportamiento contrario, donde solo tenemos respuestas de aquellos casos que son Borderlines o no reportan diabetes, y valores nulos de personas que si reportaron diabetes.

Es por esto que tomamos la decisión de, para este archivo, solo conservar el número de secuencia (`SEQN`) y la columna etiquetada (`Diabetes`)

In [ ]:
objetivo = df_etapa1[["SEQN", "Diabetes"]].copy()
objetivo

,SEQN,Diabetes
0,73557.0,1
1,73558.0,1
2,73559.0,1
3,73560.0,0
4,73561.0,0
...,...,...
9765,83727.0,0
9766,83728.0,0
9767,83729.0,0
9768,83730.0,0


### **1.5. Agregar nueva información de la NHANES 2013-2014**

Tras encontrar que el dataset seleccionado inicialmente presenta un severo desbalanceo de clases debido a que la información no estaba completa y el dataset original presenta un severo data leakage dado que muchas de las respuestas están condicionadas a presentar o no presentar diabetes, decidimos expandir la busqueda a varios de los datasets presentes en la encuesta NHANES 2013-2014 (https://wwwn.cdc.gov/nchs/nhanes/ContinuousNhanes/Default.aspx?BeginYear=2013), donde encontramos las siguientes opciones de Datasets con información que podría complementar nuestro estudio y con la opción de combinarlos haciendo uso del identificador único por paciente `SEQN`:

| Módulo | Contenido | Documentacion del Dataset | URL |
|---|---|---|---|
| DEMO_H | Demografía | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DEMO_H.htm | https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Demographics&Cycle=2013-2014 |
| BMX_H | Medidas corporales | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BMX_H.htm | https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Examination&CycleBeginYear=2013 |
| BPX_H | Presión arterial | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BPX_H.htm | https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Examination&CycleBeginYear=2013 |
| TCHOL_H | Colesterol total | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/TCHOL_H.htm | https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Laboratory&CycleBeginYear=2013 |
| HDL_H | Colesterol HDL | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/HDL_H.htm | https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Laboratory&CycleBeginYear=2013 |

Con el objetivo de identificar nuevas variables que podamos considerar para este estudio y la documentación disponible en la ágina web de NHANES, elegimos 5 bases de datos adicionales que consideramos pueden tener variables relevantes para nuestro estudio, estas son:

- DEMO_H = Demografía
- BMX_H = Medidas corporales
- BPX_H = Presión arterial
- TCHOL_H = Colesterol total
- HDL_H = Colestero HDL

In [7]:
Demografia = pd.read_sas(RUTA_RAW / "DEMO_H.xpt", format="xport")
Medidas_c = pd.read_sas(RUTA_RAW / "BMX_H.xpt", format="xport")
Presion_a = pd.read_sas(RUTA_RAW / "BPX_H.xpt", format="xport")
Colesterol_t = pd.read_sas(RUTA_RAW / "TCHOL_H.xpt", format="xport")
Colesterol_hdl = pd.read_sas(RUTA_RAW / "HDL_H.xpt", format="xport")

### **1.6. Identificación de variables dentro del Dataset Demografía**

De la documentación oficial del NHANES logramos extraer la información de cada una de las variables del Dataset **DEMO_H**, donde identificamos las siguientes variables que podrían ser interesantes para nuestro estudio:

- RIDAGEYR - Edad al momento de ser encuestado (años)
- RIAGENDR - Genero del encuestado
- RIDRETH3 - Raza/origen hispánico y asiatico
- INDHHIN2 - Ingreso anual del hogar
- INDFMIN2 - Ingreso anual de la familia

Consideramos que de las variables presentes en el dataset Demografía, las mencionadas pueden tener cierta influencia en el desarrollo de diabetes o tienen cierta capacidad de predicción. Sin embargo esta hipótesis tendremos que validarla más adelante.

In [20]:
Demografia_reducida = Demografia[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "INDHHIN2", "INDFMIN2"]].copy()

Demografia_reducida

,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,INDHHIN2,INDFMIN2
0,73557,69.0,1.0,4.0,4.0,4.0
1,73558,54.0,1.0,3.0,7.0,7.0
2,73559,72.0,1.0,3.0,10.0,10.0
3,73560,9.0,1.0,3.0,9.0,9.0
4,73561,73.0,2.0,3.0,15.0,15.0
...,...,...,...,...,...,...
10170,83727,26.0,1.0,2.0,77.0,77.0
10171,83728,2.0,2.0,1.0,8.0,8.0
10172,83729,42.0,2.0,4.0,7.0,7.0
10173,83730,7.0,1.0,2.0,6.0,6.0


### **1.7. Identificación de variables dentro del Dataset Medidas Corporales**

De la documentación oficial del NHANES logramos extraer la información de cada una de las variables del Dataset **BMX_H**, donde identificamos las siguientes variables que podrían ser interesantes para nuestro estudio:

- BMXBMI - Índice de masa corporal (kg/m**2)
- BMXWAIST - Circunferencia de la cintura (cm)

De igual manera que en Demografía, consideramos que de las variables presentes en el dataset Medidas Corporales, las mencionadas pueden tener cierta influencia en el desarrollo de diabetes o tienen cierta capacidad de predicción. Sin embargo esta hipótesis tendremos que validarla más adelante.

In [21]:
Medidas_c_reducida = Medidas_c[["SEQN", "BMXBMI", "BMXWAIST"]].copy()

Medidas_c_reducida

,SEQN,BMXBMI,BMXWAIST
0,73557,26.7,100.0
1,73558,28.6,107.6
2,73559,28.9,109.2
3,73560,17.1,61.0
4,73561,19.7,NaN
...,...,...,...
9808,83727,24.5,87.1
9809,83728,15.9,47.2
9810,83729,34.0,102.7
9811,83730,16.1,53.0


### **1.8. Identificación de variables dentro del Dataset Presion Arterial**

Conforme a la documentación, identificamos las siguientes variables predictoras potenciales: 

- BPXSY1 - Sistólica: presión arterial (1ra lectura) mm Hg
- BPXDI1 - Diastólica: presión arterial (1ra lectura) mm Hg


In [22]:
Presion_a_reducida = Presion_a[["SEQN", "BPXSY1", "BPXDI1"]].copy()

Presion_a_reducida

,SEQN,BPXSY1,BPXDI1
0,73557,122.0,72.0
1,73558,156.0,62.0
2,73559,140.0,90.0
3,73560,108.0,38.0
4,73561,136.0,86.0
...,...,...,...
9808,83727,110.0,68.0
9809,83728,NaN,NaN
9810,83729,136.0,82.0
9811,83730,NaN,NaN


### **1.9. Identificación de variables dentro del Dataset Colesterol Total**

Conforme a la documentación, identificamos las siguientes variables predictoras potenciales: 

- LBXTC - Colesterol Total ( mg/dL)

In [23]:
Colesterol_t_reducida = Colesterol_t[["SEQN", "LBXTC"]].copy()

Colesterol_t_reducida

,SEQN,LBXTC
0,73557,167.0
1,73558,170.0
2,73559,126.0
3,73560,168.0
4,73561,201.0
...,...,...
8286,83726,NaN
8287,83727,190.0
8288,83729,152.0
8289,83730,167.0


### **1.10. Identificación de variables dentro del Dataset Colesterol HDL**

Conforme a la documentación, identificamos las siguientes variables predictoras potenciales: 

- LBDHDD - Colesterol Total ( mg/dL)

In [24]:
Colesterol_hdl_reducida = Colesterol_hdl[["SEQN", "LBDHDD"]].copy()

Colesterol_hdl_reducida

,SEQN,LBDHDD
0,73557,65.0
1,73558,50.0
2,73559,60.0
3,73560,61.0
4,73561,85.0
...,...,...
8286,83726,NaN
8287,83727,55.0
8288,83729,48.0
8289,83730,51.0


### **1.11. combinación de los Datasets y las variables seleccionadas**

Una vez seleccionadas las variables predictoras potenciales que usaremos en nuestro estudio, procedemos a unir todos los registros en un solo dataset por medio del número de secuencia.

In [25]:
tablas = [
    Demografia_reducida,
    Medidas_c_reducida,
    Presion_a_reducida,
    Colesterol_t_reducida,
    Colesterol_hdl_reducida,
]

for t in [objetivo] + tablas:
    t["SEQN"] = t["SEQN"].astype(int)

df_consolidado = objetivo.copy()

for t in tablas:
    df_consolidado = df_consolidado.merge(t, on="SEQN", how="left")

# Verificaciones respectivas para asegurar que no se perdieron y tampoco se duplicaron filas durante la unión de tablas
print(df_consolidado.shape)
print(len(df_consolidado) == len(objetivo))
print(df_consolidado["SEQN"].is_unique)

df_consolidado

(9763, 13)
True
True


,SEQN,Diabetes,RIDAGEYR,RIAGENDR,RIDRETH3,INDHHIN2,INDFMIN2,BMXBMI,BMXWAIST,BPXSY1,BPXDI1,LBXTC,LBDHDD
0,73557,1,69.0,1.0,4.0,4.0,4.0,26.7,100.0,122.0,72.0,167.0,65.0
1,73558,1,54.0,1.0,3.0,7.0,7.0,28.6,107.6,156.0,62.0,170.0,50.0
2,73559,1,72.0,1.0,3.0,10.0,10.0,28.9,109.2,140.0,90.0,126.0,60.0
3,73560,0,9.0,1.0,3.0,9.0,9.0,17.1,61.0,108.0,38.0,168.0,61.0
4,73561,0,73.0,2.0,3.0,15.0,15.0,19.7,NaN,136.0,86.0,201.0,85.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9758,83727,0,26.0,1.0,2.0,77.0,77.0,24.5,87.1,110.0,68.0,190.0,55.0
9759,83728,0,2.0,2.0,1.0,8.0,8.0,15.9,47.2,NaN,NaN,NaN,NaN
9760,83729,0,42.0,2.0,4.0,7.0,7.0,34.0,102.7,136.0,82.0,152.0,48.0
9761,83730,0,7.0,1.0,2.0,6.0,6.0,16.1,53.0,NaN,NaN,167.0,51.0


In [29]:
df_consolidado.to_csv(RUTA_RAW / "NHANES_consolidado.csv", index=False)

## **2. Exploración del nuevo Conjunto de Datos a utilizar**

